In [32]:
pip install transformers peft accelerate datasets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer,AutoModelForSequenceClassification,Trainer,TrainingArguments
from peft import LoraConfig,get_peft_model


In [34]:
torch.__version__
torch.cuda.is_available()
print("done")

done


In [35]:
tokenizer=AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
model=AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased",
    num_labels=3
    
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 318.06it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [36]:
lora_config=LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_lin","v_lin"],
    lora_dropout=0.1,
    task_type="SEQ_CLS"
)

In [37]:
model=get_peft_model(model,lora_config)
model.print_trainable_parameters()


trainable params: 740,355 || all params: 67,696,134 || trainable%: 1.0936


In [38]:
print("DistilBERT+ Lora ready")

DistilBERT+ Lora ready


In [39]:
df1=pd.read_csv(r"C:\Users\LENEVO\Downloads\test.csv",encoding="latin1")

In [40]:
df1.columns.tolist()
df1.head(2)

,textID,text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,f87dea47db,Last session of the day http://twitpic.com/67ezh,neutral,morning,0-20,Afghanistan,38928346.0,652860.0,60.0
1,96d74cb729,Shanghai is also really exciting (precisely -...,positive,noon,21-30,Albania,2877797.0,27400.0,105.0


In [41]:
label_map={"negative":0,"neutral":1,"positive":2}
df1["label"]=df1["sentiment"].map(label_map)



In [42]:
label_map

{'negative': 0, 'neutral': 1, 'positive': 2}

In [43]:
df=df1[["text","label"]].dropna().sample(2000,random_state=42)


In [44]:
df

,text,label
325,Thank you so much phaoloo !!!!,2.0
2226,Midnight ice-cream weather! So **** bored,0.0
2480,Ohh i forgot to tell you last night that when ...,2.0
1862,i am the only arabic girl who`s online every ...,1.0
299,starting the video editing of the first spanki...,1.0
...,...,...
2064,"Beyonce helps me through the day, I`m starting...",1.0
3186,me too. ****.,0.0
3150,YES! I DONE IT! at last. my homework is ready...,2.0
3164,"I can only imagine, but I know I love hearing...",2.0


In [45]:
from sklearn.model_selection import train_test_split
train_df,test_df=train_test_split(df,test_size=0.2,random_state=42)

In [46]:
def tokenize(batch):
    return tokenizer(batch["text"],padding="max_length",truncation=True,max_length=128)
train_ds=Dataset.from_pandas(train_df)
test_ds=Dataset.from_pandas(test_df)
train_df=train_ds.map(tokenize,batched=True)
test_df=test_ds.map(tokenize,batched=True)
len(train_ds)
len(test_ds)
print("data ready")

Map: 100%|██████████| 400/400 [00:00<00:00, 1986.23 examples/s]

data ready


In [48]:

model.config.num_labels = 3
model.config.problem_type = "single_label_classification"
model.config.id2label = {0: 'negative', 1: 'neutral', 2: 'positive'}
model.config.label2id = {'negative': 0, 'neutral': 1, 'positive': 2}

print("Fixed! Model")

Fixed! Model


In [53]:

from datasets import ClassLabel

train_df = train_df.cast_column("label", ClassLabel(num_classes=3, names=['negative','neutral','positive']))
test_df = test_df.cast_column("label", ClassLabel(num_classes=3, names=['negative','neutral','positive']))

print("Label Fixed! এখন 0, 1, 2 আস্ত সংখ্যা")
print(train_df[0])

Casting the dataset: 100%|██████████| 400/400 [00:00<00:00, 33132.98 examples/s]

Label Fixed! এখন 0, 1, 2 আস্ত সংখ্যা
{'label': 2, 'input_ids': [101, 18783, 1012, 1998, 1045, 2134, 1036, 1056, 2131, 2008, 27263, 20917, 4014, 2022, 3772, 1996, 7966, 2823, 1012, 2064, 1057, 1040, 2213, 2033, 2009, 1029, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

In [54]:


args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none"
)

def compute_metrics(p):
    import numpy as np
    preds = np.argmax(p.predictions, axis=1)
    from sklearn.metrics import accuracy_score
    return {"accuracy": accuracy_score(p.label_ids, preds)}

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_df,
    eval_dataset=test_df,
    compute_metrics=compute_metrics
)

trainer.train()
print("Training Done!")

Epoch,Training Loss,Validation Loss,Accuracy
1,1.006797,0.990822,0.510000
2,0.808190,0.842205,0.640000


c:\Users\LENEVO\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Done!


In [55]:
trainer.evaluate()

c:\Users\LENEVO\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
0.808190,0.842205,2,0.640000


{'eval_loss': 0.8422048091888428, 'eval_accuracy': 0.64}

In [56]:

from transformers import pipeline

clf = pipeline("text-classification", model=model, tokenizer=tokenizer)

print(clf("I love this movie, it was amazing!"))
print(clf("This is terrible, I hate it"))
print(clf("It was okay, not good not bad"))

[{'label': 'positive', 'score': 0.7659566402435303}]
[{'label': 'negative', 'score': 0.6815196871757507}]
[{'label': 'negative', 'score': 0.49189215898513794}]


In [57]:
# মডেল সেভ করা
model.save_pretrained("./my-sentiment-model")
tokenizer.save_pretrained("./my-sentiment-model")
print("Model Saved! এখন GitHub এ দেওয়া যাবে")

Model Saved! এখন GitHub এ দেওয়া যাবে
